# BTC/USDT 6-Hour-Ahead (12-Bar) Direction Prediction

Predict whether the **open price 12 bars ahead (6 hours later)** will be
higher or lower than the **current bar's open price**, and by how many
percent, using a LightGBM model trained on engineered technical features
from the 30-minute BTC/USDT bar data.

This is the same pipeline as the 1-bar-ahead notebook (`model.ipynb`), with
the forecast horizon extended from 1 bar (30 minutes) to 12 bars (6 hours).

**Target**
- `y_cls`: 1 if `open[t+12] > open[t]`, else 0 (binary direction)
- `y_reg`: `(open[t+12] - open[t]) / open[t] * 100` (signed percent move over the next 12 bars)

**Leakage guard**: the target window `[t, t+12]` starts with bar `t`'s own
price action (since `open[t+1]` is essentially bar `t`'s close), so features
used to predict it are still built from data available **through bar
`t-1`** only (the feature matrix is shifted forward by one bar relative to
the target) — no information from bar `t` or later ever reaches the model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, mean_absolute_error, mean_squared_error,
)

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

In [2]:
dataset_lag1 = pd.read_csv("./data/processed_dataset_target_lag1.csv", index_col=0)
dataset_lag6 = pd.read_csv("./data/processed_dataset_target_lag6.csv", index_col=0)
dataset_lag12 = pd.read_csv("./data/processed_dataset_target_lag12.csv", index_col=0)

In [ ]:
feature_cols = [col for col in dataset_lag1.columns if col not in ("y_reg")]

## 1. Time-based train / validation / test split

Chronological split (no shuffling) — 70% train / 15% validation / 15% test —
since shuffling would leak future information into training for a time series.

In [4]:
X = dataset_lag12[feature_cols]
# y_cls = dataset_lag12["y_cls"]
y_reg = dataset_lag12["y_reg"]

n = len(dataset_lag12)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, X_val, X_test = X.iloc[:train_end], X.iloc[train_end:val_end], X.iloc[val_end:]
# yc_train, yc_val, yc_test = y_cls.iloc[:train_end], y_cls.iloc[train_end:val_end], y_cls.iloc[val_end:]
yr_train, yr_val, yr_test = y_reg.iloc[:train_end], y_reg.iloc[train_end:val_end], y_reg.iloc[val_end:]

print("train:", X_train.shape, dataset_lag12.index[0], "->", dataset_lag12.index[train_end - 1])
print("val:  ", X_val.shape, dataset_lag12.index[train_end], "->", dataset_lag12.index[val_end - 1])
print("test: ", X_test.shape, dataset_lag12.index[val_end], "->", dataset_lag12.index[-1])

train: (36677, 72) 2023-09-06 04:00:00+00:00 -> 2025-10-09 06:00:00+00:00
val:   (7859, 72) 2025-10-09 06:30:00+00:00 -> 2026-03-21 23:30:00+00:00
test:  (7860, 72) 2026-03-22 00:00:00+00:00 -> 2026-09-01 17:30:00+00:00


## 8. Magnitude regression

A second LightGBM regressor predicts the signed percent move itself
(`y_reg`), so predictions can be reported as "up/down by X% over the next
12 bars" rather than direction alone. Compared against a naive
baseline that always predicts 0%.

In [12]:
reg = lgb.LGBMRegressor(
    n_estimators=3000, min_child_samples=50, subsample=0.8, subsample_freq=1,
    reg_alpha=0.1, reg_lambda=0.1, objective="regression",
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1, **best_params,
)
reg.fit(
    X_train, yr_train,
    eval_set=[(X_val, yr_val)], eval_metric="l2",
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
)

pred_reg = reg.predict(X_test)
mae = mean_absolute_error(yr_test, pred_reg)
rmse = mean_squared_error(yr_test, pred_reg) ** 0.5
naive_mae = mean_absolute_error(yr_test, np.zeros_like(yr_test))
dir_acc_from_reg = ((pred_reg > 0).astype(int) == yc_test.values).mean()

print(f"MAE:  {mae:.4f}%")
print(f"RMSE: {rmse:.4f}%")
print(f"Naive (predict 0%) MAE: {naive_mae:.4f}%")
print(f"Direction accuracy implied by regressor sign: {dir_acc_from_reg:.4f}")

/Users/suppanatp/Documents/NUS FE/2 Courses/FE5209 FINANCIAL ECONOMETRICS/Project/AlgoGoneCrazy/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


MAE:  0.6732%
RMSE: 0.9743%
Naive (predict 0%) MAE: 0.6725%
Direction accuracy implied by regressor sign: 0.5160


## 10. Discussion\n\n_placeholder - filled in after execution based on actual results_